In [1]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [9]:
ferrao_query = f"""select
    date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
    customerid,
    sum(ao_session) ao_session,
    sum(fe_session) fe_session,
    sum(rr_session) rr_session,
    sum(ao_day) ao_day,
    sum(fe_day) fe_day,
    sum(rr_day) rr_day
from
(
    select 
        day,
        customerid, 
        max(ao_sessions_unique_daily) ao_session, 
        max(fe_sessions_unique_daily) fe_session, 
        sum(rr_sessions_unique_daily) rr_session, 
        max(case when ao_sessions_unique_daily > 0 then 1 else 0 end) ao_day,
        max(case when fe_sessions_unique_daily > 0 then 1 else 0 end) fe_day,
        max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_day
    from 
        datasets.customer_rf_daily_kpi
    where 
        day >= '2024-12-16'
        and day <= '2025-03-16'
        and city = 'Bangalore'
        and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    order by 1
)
group by 1,2"""

In [11]:
# ferrao_df = pd.read_sql(ferrao_query, conn3)

In [13]:
# ferrao_df.to_csv('ferrao_df.csv', index=False, header=False)

In [15]:
# ferrao_df.columns

In [17]:
# ferrao_df

### new segment

In [21]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        segment,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.6,4.7,4.5,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.6,4.8,4.6,4.2,3.6,3.8,4.5,4.2,4.6,4.6,4.7,4.7,4.6,4.4,4.5,4.3
1,2-WEEKLY,2.4,2.2,2.3,2.5,2.4,2.5,2.5,2.6,2.6,2.5,2.5,2.6,2.5,2.4,2.3,2.3,2.5,2.4,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5,2.1,1.9,2.0,2.2,2.1,2.2,2.2,2.3,2.3,2.2,2.2,2.3,2.2
2,3-MONTHLY,1.6,1.6,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3
3,4-OTHER,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.5,1.6,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.6,1.6,0.9,0.8,0.8,0.7,0.7,0.7,0.7,0.7,0.9,1.0,1.0,1.1,1.1
4,5-INACTIVE,0.7,0.7,0.7,0.7,0.7,0.7,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.8,0.7,0.7,0.7,0.7,0.7,0.7,0.6,0.6,0.6,0.6,0.6,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
cx_data0

,segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,5.2,4.5,4.7,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.6,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.5,5.3,5.4,5.2,5.0,4.4,4.6,5.3,4.9,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
1,2-WEEKLY,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.8,2.9,2.8,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.9,2.8,2.9,2.8,2.4,2.1,2.2,2.5,2.4,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5
2,3-MONTHLY,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3
3,4-OTHER,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.5,1.6,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.6,1.6,0.9,0.8,0.8,0.7,0.7,0.7,0.7,0.7,0.9,1.0,1.0,1.1,1.1
4,5-INACTIVE,0.7,0.7,0.7,0.7,0.7,0.7,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.8,0.7,0.7,0.7,0.7,0.7,0.7,0.6,0.6,0.6,0.6,0.6,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Office Bins

In [2]:
cx_data15 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        office_bin,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,a. no-office-transit,1.5,1.4,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.6,1.6,1.0,0.9,0.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.1,1.1
1,b. office ride,2.2,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.3,2.3,2.2,2.3,2.3,2.2,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.3,2.3,2.3,2.3,2.3,1.9,1.7,1.7,1.9,1.8,1.9,1.9,2.0,2.0,1.9,1.9,2.0,1.9
2,c. transit ride,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.3,1.3,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.4,1.4,1.4


In [6]:
cx_data15

,office_bin,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,a. no-office-transit,1.5,1.4,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.6,1.6,1.0,0.9,0.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.1,1.1
1,b. office ride,2.2,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.3,2.3,2.2,2.3,2.3,2.2,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.3,2.3,2.3,2.3,2.3,1.9,1.7,1.7,1.9,1.8,1.9,1.9,2.0,2.0,1.9,1.9,2.0,1.9
2,c. transit ride,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.3,1.3,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.4,1.4,1.4


In [7]:
cx_data15.to_clipboard(index=False)

In [4]:
cx_data16 = f'''
    with base as (
    
    select
        customer_id,
        segment,
        -- total_net_orders,
        -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        segment,
        office_bin,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,a. no-office-transit,4.4,3.7,4.0,4.7,4.3,4.9,4.8,4.8,4.8,4.8,4.5,4.7,4.5,4.4,3.8,4.0,4.7,4.3,4.9,4.9,4.9,4.8,4.8,4.5,4.7,4.5,4.2,3.5,3.7,4.5,4.1,4.7,4.6,4.6,4.6,4.5,4.3,4.5,4.3
1,1-DAILY,b. office ride,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.6,4.7,4.5,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.7,4.8,4.6,4.3,3.7,3.8,4.5,4.2,4.6,4.6,4.7,4.7,4.6,4.4,4.6,4.3
2,1-DAILY,c. transit ride,4.4,3.9,4.1,4.7,4.3,4.8,4.8,4.8,4.8,4.8,4.5,4.7,4.6,4.4,3.9,4.1,4.7,4.4,4.8,4.8,4.9,4.8,4.8,4.6,4.7,4.6,4.2,3.6,3.8,4.5,4.1,4.6,4.6,4.6,4.6,4.6,4.3,4.5,4.4
3,2-WEEKLY,a. no-office-transit,2.3,2.1,2.2,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.3,2.4,2.4,2.4,2.2,2.2,2.4,2.3,2.5,2.4,2.5,2.5,2.4,2.3,2.4,2.4,2.0,1.8,1.8,2.1,2.0,2.1,2.1,2.2,2.1,2.1,2.0,2.1,2.1
4,2-WEEKLY,b. office ride,2.5,2.3,2.3,2.6,2.4,2.6,2.6,2.6,2.7,2.6,2.5,2.6,2.6,2.5,2.3,2.3,2.6,2.4,2.6,2.6,2.7,2.7,2.6,2.6,2.6,2.6,2.2,1.9,2.0,2.2,2.1,2.3,2.2,2.3,2.4,2.3,2.2,2.3,2.3


In [5]:
cx_data16

,segment,office_bin,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,a. no-office-transit,4.4,3.7,4.0,4.7,4.3,4.9,4.8,4.8,4.8,4.8,4.5,4.7,4.5,4.4,3.8,4.0,4.7,4.3,4.9,4.9,4.9,4.8,4.8,4.5,4.7,4.5,4.2,3.5,3.7,4.5,4.1,4.7,4.6,4.6,4.6,4.5,4.3,4.5,4.3
1,1-DAILY,b. office ride,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.6,4.7,4.5,4.5,3.9,4.0,4.7,4.4,4.8,4.8,4.9,4.9,4.8,4.7,4.8,4.6,4.3,3.7,3.8,4.5,4.2,4.6,4.6,4.7,4.7,4.6,4.4,4.6,4.3
2,1-DAILY,c. transit ride,4.4,3.9,4.1,4.7,4.3,4.8,4.8,4.8,4.8,4.8,4.5,4.7,4.6,4.4,3.9,4.1,4.7,4.4,4.8,4.8,4.9,4.8,4.8,4.6,4.7,4.6,4.2,3.6,3.8,4.5,4.1,4.6,4.6,4.6,4.6,4.6,4.3,4.5,4.4
3,2-WEEKLY,a. no-office-transit,2.3,2.1,2.2,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.3,2.4,2.4,2.4,2.2,2.2,2.4,2.3,2.5,2.4,2.5,2.5,2.4,2.3,2.4,2.4,2.0,1.8,1.8,2.1,2.0,2.1,2.1,2.2,2.1,2.1,2.0,2.1,2.1
4,2-WEEKLY,b. office ride,2.5,2.3,2.3,2.6,2.4,2.6,2.6,2.6,2.7,2.6,2.5,2.6,2.6,2.5,2.3,2.3,2.6,2.4,2.6,2.6,2.7,2.7,2.6,2.6,2.6,2.6,2.2,1.9,2.0,2.2,2.1,2.3,2.2,2.3,2.4,2.3,2.2,2.3,2.3
5,2-WEEKLY,c. transit ride,2.3,2.2,2.3,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.4,2.4,2.4,2.4,2.2,2.3,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.4,2.4,2.4,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.1,2.1
6,3-MONTHLY,a. no-office-transit,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.6,1.5,1.5,1.5,1.6,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.6,1.6,1.5,1.5,1.6,1.6,1.1,1.0,1.0,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1
7,3-MONTHLY,b. office ride,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.4,1.3,1.3,1.4,1.3,1.4,1.4,1.5,1.5,1.4,1.4,1.5,1.5
8,3-MONTHLY,c. transit ride,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.6,1.6,1.6,1.7,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.7,1.6,1.6,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
9,4-OTHER,a. no-office-transit,1.3,1.3,1.3,1.3,1.2,1.3,1.2,1.2,1.3,1.4,1.4,1.4,1.5,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4,1.4,1.5,0.8,0.8,0.7,0.7,0.7,0.7,0.7,0.7,0.8,0.8,0.9,0.9,1.0


In [ ]:
cx_data16.to_clipboard(index=False)

In [ ]:
cx_data0.to_clipboard(index=False)

In [6]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2 
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,NRHA,5.1,4.8,5.0,5.4,5.2,5.5,5.5,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.5,5.5,5.6,5.6,5.5,5.3,5.4,5.3,5.0,4.6,4.9,5.3,5.1,5.4,5.4,5.4,5.4,5.4,5.2,5.3,5.1
1,1-DAILY,RHA,5.2,4.5,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.6,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.5,5.3,5.4,5.2,5.0,4.3,4.6,5.3,4.9,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
2,1-DAILY,UNKNOWN,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0
3,2-WEEKLY,NRHA,2.5,2.4,2.5,2.7,2.6,2.7,2.6,2.7,2.7,2.7,2.6,2.7,2.6,2.5,2.4,2.5,2.7,2.6,2.7,2.7,2.8,2.8,2.7,2.6,2.7,2.6,2.2,2.1,2.2,2.4,2.3,2.4,2.4,2.4,2.5,2.4,2.3,2.4,2.3
4,2-WEEKLY,RHA,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.8,2.9,2.8,2.7,2.5,2.5,2.8,2.7,2.9,2.9,2.9,2.9,2.9,2.8,2.9,2.8,2.4,2.1,2.2,2.5,2.3,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5


In [7]:
cx_data11

,segment,RHA_Check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,NRHA,5.1,4.8,5.0,5.4,5.2,5.5,5.5,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.5,5.5,5.6,5.6,5.5,5.3,5.4,5.3,5.0,4.6,4.9,5.3,5.1,5.4,5.4,5.4,5.4,5.4,5.2,5.3,5.1
1,1-DAILY,RHA,5.2,4.5,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.6,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.5,5.3,5.4,5.2,5.0,4.3,4.6,5.3,4.9,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
2,1-DAILY,UNKNOWN,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0
3,2-WEEKLY,NRHA,2.5,2.4,2.5,2.7,2.6,2.7,2.6,2.7,2.7,2.7,2.6,2.7,2.6,2.5,2.4,2.5,2.7,2.6,2.7,2.7,2.8,2.8,2.7,2.6,2.7,2.6,2.2,2.1,2.2,2.4,2.3,2.4,2.4,2.4,2.5,2.4,2.3,2.4,2.3
4,2-WEEKLY,RHA,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.8,2.9,2.8,2.7,2.5,2.5,2.8,2.7,2.9,2.9,2.9,2.9,2.9,2.8,2.9,2.8,2.4,2.1,2.2,2.5,2.3,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5
5,2-WEEKLY,UNKNOWN,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.8,3.0,2.8,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.9,3.0,2.9,2.5,2.1,2.2,2.6,2.4,2.7,2.6,2.7,2.7,2.6,2.6,2.7,2.6
6,3-MONTHLY,NRHA,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.7,1.6,1.6,1.6,1.7,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.7,1.6,1.6,1.6,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
7,3-MONTHLY,RHA,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3
8,3-MONTHLY,UNKNOWN,1.7,1.6,1.6,1.7,1.6,1.7,1.7,1.8,1.8,1.7,1.7,1.8,1.8,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.8,1.8,1.8,1.7,1.8,1.8,1.3,1.2,1.2,1.3,1.3,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4
9,4-OTHER,NRHA,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4,1.5,1.5,1.5,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.5,1.6,0.9,0.8,0.8,0.8,0.7,0.8,0.7,0.7,0.8,0.9,1.0,1.0,1.1


In [10]:
cx_data11.to_clipboard(index=False)

In [8]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,Non-Zwigato,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.4,5.4,5.3,5.0,4.6,4.8,5.3,5.0,5.4,5.4,5.4,5.4,5.3,5.2,5.2,5.1
1,1-DAILY,UNKNOWN,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0
2,1-DAILY,Zwigato,5.2,4.5,4.8,5.4,5.0,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.6,5.3,4.9,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
3,2-WEEKLY,Non-Zwigato,2.6,2.5,2.5,2.8,2.6,2.8,2.7,2.8,2.8,2.8,2.7,2.7,2.7,2.6,2.5,2.5,2.8,2.6,2.8,2.8,2.8,2.9,2.8,2.7,2.8,2.7,2.3,2.1,2.2,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.3,2.4,2.3
4,2-WEEKLY,UNKNOWN,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.8,3.0,2.8,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.9,3.0,2.9,2.5,2.1,2.2,2.6,2.4,2.7,2.6,2.7,2.7,2.6,2.6,2.7,2.6


In [9]:
cx_data12

,segment,Zwigato_check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,Non-Zwigato,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.4,5.4,5.3,5.0,4.6,4.8,5.3,5.0,5.4,5.4,5.4,5.4,5.3,5.2,5.2,5.1
1,1-DAILY,UNKNOWN,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0
2,1-DAILY,Zwigato,5.2,4.5,4.8,5.4,5.0,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.8,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.6,5.3,4.9,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
3,2-WEEKLY,Non-Zwigato,2.6,2.5,2.5,2.8,2.6,2.8,2.7,2.8,2.8,2.8,2.7,2.7,2.7,2.6,2.5,2.5,2.8,2.6,2.8,2.8,2.8,2.9,2.8,2.7,2.8,2.7,2.3,2.1,2.2,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.3,2.4,2.3
4,2-WEEKLY,UNKNOWN,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.8,3.0,2.8,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.9,3.0,2.9,2.5,2.1,2.2,2.6,2.4,2.7,2.6,2.7,2.7,2.6,2.6,2.7,2.6
5,2-WEEKLY,Zwigato,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.8,2.9,2.8,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.8,2.9,2.8,2.4,2.1,2.2,2.5,2.4,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5
6,3-MONTHLY,Non-Zwigato,1.6,1.6,1.6,1.7,1.6,1.7,1.6,1.7,1.7,1.7,1.6,1.7,1.7,1.6,1.6,1.6,1.7,1.6,1.7,1.6,1.7,1.7,1.7,1.6,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
7,3-MONTHLY,UNKNOWN,1.7,1.6,1.6,1.7,1.6,1.7,1.7,1.8,1.8,1.7,1.7,1.8,1.8,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.8,1.8,1.8,1.7,1.8,1.8,1.3,1.2,1.2,1.3,1.3,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4
8,3-MONTHLY,Zwigato,1.6,1.6,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3
9,4-OTHER,Non-Zwigato,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.5,1.6,1.4,1.4,1.4,1.3,1.3,1.3,1.3,1.3,1.4,1.5,1.5,1.5,1.6,0.9,0.8,0.8,0.7,0.7,0.7,0.7,0.7,0.8,0.9,0.9,1.0,1.1


In [11]:
cx_data12.to_clipboard(index=False)

In [12]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,1. BOTH,5.2,4.5,4.7,5.4,5.0,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.7,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.5,5.2,4.9,5.3,5.4,5.4,5.4,5.3,5.1,5.2,5.0
1,1-DAILY,2. ONLY Zwigato,5.1,4.7,5.0,5.4,5.1,5.5,5.4,5.5,5.5,5.5,5.3,5.4,5.2,5.2,4.7,5.0,5.4,5.2,5.5,5.5,5.5,5.5,5.5,5.3,5.4,5.2,5.0,4.6,4.8,5.3,5.0,5.4,5.3,5.4,5.4,5.4,5.2,5.3,5.1
2,1-DAILY,3. ONLY RHA,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.6,5.4,5.4,5.3,5.0,4.6,4.8,5.3,5.0,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
3,1-DAILY,4. NONE,5.1,4.8,5.0,5.5,5.2,5.6,5.5,5.6,5.6,5.5,5.3,5.4,5.3,5.2,4.8,5.0,5.5,5.2,5.6,5.5,5.6,5.6,5.5,5.4,5.4,5.3,5.0,4.6,4.9,5.3,5.1,5.4,5.4,5.5,5.5,5.4,5.2,5.3,5.1
4,1-DAILY,5. IOS,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0


In [13]:
cx_data13

,segment,tech_savvy,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1-DAILY,1. BOTH,5.2,4.5,4.7,5.4,5.0,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.7,5.4,5.1,5.5,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.5,5.2,4.9,5.3,5.4,5.4,5.4,5.3,5.1,5.2,5.0
1,1-DAILY,2. ONLY Zwigato,5.1,4.7,5.0,5.4,5.1,5.5,5.4,5.5,5.5,5.5,5.3,5.4,5.2,5.2,4.7,5.0,5.4,5.2,5.5,5.5,5.5,5.5,5.5,5.3,5.4,5.2,5.0,4.6,4.8,5.3,5.0,5.4,5.3,5.4,5.4,5.4,5.2,5.3,5.1
2,1-DAILY,3. ONLY RHA,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.5,5.3,5.4,5.2,5.2,4.8,5.0,5.5,5.2,5.6,5.6,5.6,5.6,5.6,5.4,5.4,5.3,5.0,4.6,4.8,5.3,5.0,5.4,5.4,5.4,5.4,5.3,5.1,5.2,5.0
3,1-DAILY,4. NONE,5.1,4.8,5.0,5.5,5.2,5.6,5.5,5.6,5.6,5.5,5.3,5.4,5.3,5.2,4.8,5.0,5.5,5.2,5.6,5.5,5.6,5.6,5.5,5.4,5.4,5.3,5.0,4.6,4.9,5.3,5.1,5.4,5.4,5.5,5.5,5.4,5.2,5.3,5.1
4,1-DAILY,5. IOS,5.1,4.4,4.6,5.4,5.0,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.2,4.5,4.6,5.4,5.1,5.4,5.5,5.5,5.5,5.4,5.3,5.4,5.2,5.0,4.3,4.4,5.3,4.9,5.3,5.4,5.4,5.4,5.3,5.2,5.3,5.0
5,2-WEEKLY,1. BOTH,2.7,2.4,2.5,2.8,2.7,2.9,2.8,2.9,2.9,2.9,2.8,2.9,2.8,2.7,2.4,2.5,2.9,2.7,2.9,2.9,2.9,3.0,2.9,2.8,2.9,2.8,2.4,2.1,2.2,2.5,2.4,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.5
6,2-WEEKLY,2. ONLY Zwigato,2.5,2.4,2.5,2.6,2.5,2.7,2.6,2.7,2.7,2.7,2.6,2.6,2.6,2.5,2.4,2.5,2.7,2.6,2.7,2.6,2.7,2.7,2.7,2.6,2.7,2.6,2.2,2.1,2.2,2.4,2.3,2.4,2.4,2.5,2.5,2.4,2.3,2.4,2.4
7,2-WEEKLY,3. ONLY RHA,2.7,2.5,2.6,2.8,2.7,2.8,2.8,2.9,2.9,2.8,2.7,2.8,2.7,2.7,2.5,2.6,2.8,2.7,2.9,2.8,2.9,2.9,2.8,2.7,2.8,2.8,2.3,2.1,2.2,2.4,2.3,2.5,2.4,2.5,2.5,2.4,2.3,2.4,2.4
8,2-WEEKLY,4. NONE,2.5,2.4,2.5,2.7,2.6,2.7,2.7,2.7,2.8,2.7,2.6,2.7,2.6,2.6,2.4,2.5,2.7,2.6,2.7,2.7,2.8,2.8,2.7,2.6,2.7,2.6,2.2,2.1,2.2,2.4,2.3,2.4,2.4,2.4,2.4,2.4,2.3,2.4,2.3
9,2-WEEKLY,5. IOS,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.8,3.0,2.8,2.8,2.4,2.5,2.9,2.7,2.9,2.9,3.0,3.0,2.9,2.9,3.0,2.9,2.5,2.1,2.2,2.6,2.4,2.7,2.6,2.7,2.7,2.6,2.6,2.7,2.6


In [17]:
cx_data13.to_clipboard(index=False)

In [14]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn1)
cx_data14.head()

,tech_savvy,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1. BOTH,1.9,1.8,1.8,1.9,1.9,2.0,1.9,2.0,2.0,2.0,1.9,2.0,2.0,1.9,1.8,1.8,2.0,1.9,2.0,2.0,2.0,2.0,2.0,1.9,2.0,2.0,1.5,1.3,1.4,1.5,1.5,1.6,1.5,1.6,1.6,1.5,1.5,1.6,1.6
1,2. ONLY Zwigato,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.8,1.7,1.7,1.8,1.8,1.7,1.7,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.7,1.8,1.8,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4
2,3. ONLY RHA,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.2,1.2,1.2,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3
3,4. NONE,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
4,5. IOS,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8


In [15]:
cx_data14

,tech_savvy,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,1. BOTH,1.9,1.8,1.8,1.9,1.9,2.0,1.9,2.0,2.0,2.0,1.9,2.0,2.0,1.9,1.8,1.8,2.0,1.9,2.0,2.0,2.0,2.0,2.0,1.9,2.0,2.0,1.5,1.3,1.4,1.5,1.5,1.6,1.5,1.6,1.6,1.5,1.5,1.6,1.6
1,2. ONLY Zwigato,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.8,1.7,1.7,1.8,1.8,1.7,1.7,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.7,1.8,1.8,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4
2,3. ONLY RHA,1.7,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.2,1.2,1.2,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3
3,4. NONE,1.6,1.6,1.6,1.7,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.1,1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
4,5. IOS,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8


In [18]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [19]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        taxi_regularity_segment regularity_segment,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
'''
cx_data1 = pd.read_sql(cx_data, conn1)
cx_data1.head()

,regularity_segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,BI_WEEKLY,1.6,1.6,1.6,1.6,1.6,1.6,1.5,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.2,1.1,1.1,1.2,1.1,1.2,1.1,1.1,1.2,1.1,1.2,1.2,1.2
1,RARE_NEED,1.2,1.1,1.1,1.0,1.0,1.0,0.9,0.9,0.9,0.9,0.9,0.9,1.0,1.2,1.1,1.1,1.1,1.0,1.0,0.9,1.0,1.0,1.0,0.9,0.9,1.0,0.8,0.6,0.5,0.5,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4
2,MONTHLY,1.5,1.5,1.4,1.4,1.4,1.4,1.3,1.4,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.3,1.4,1.4,1.0,1.0,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.8,0.8,0.9,1.0
3,DAILY,3.7,3.3,3.4,3.9,3.7,4.0,4.0,4.1,4.1,4.0,3.9,4.0,3.9,3.8,3.3,3.4,4.0,3.7,4.0,4.0,4.1,4.1,4.0,3.9,4.1,3.9,3.5,3.0,3.1,3.7,3.4,3.7,3.7,3.8,3.8,3.7,3.6,3.8,3.6
4,WEEKLY,1.9,1.8,1.9,2.0,1.9,2.0,1.9,2.0,2.0,2.0,2.0,2.0,2.0,1.9,1.9,1.9,2.0,1.9,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.6,1.5,1.5,1.6,1.5,1.6,1.6,1.7,1.7,1.6,1.6,1.7,1.7


In [20]:
cx_data1

,regularity_segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,BI_WEEKLY,1.6,1.6,1.6,1.6,1.6,1.6,1.5,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.6,1.7,1.2,1.1,1.1,1.2,1.1,1.2,1.1,1.1,1.2,1.1,1.2,1.2,1.2
1,RARE_NEED,1.2,1.1,1.1,1.0,1.0,1.0,0.9,0.9,0.9,0.9,0.9,0.9,1.0,1.2,1.1,1.1,1.1,1.0,1.0,0.9,1.0,1.0,1.0,0.9,0.9,1.0,0.8,0.6,0.5,0.5,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4
2,MONTHLY,1.5,1.5,1.4,1.4,1.4,1.4,1.3,1.4,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.3,1.4,1.4,1.0,1.0,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.8,0.8,0.9,1.0
3,DAILY,3.7,3.3,3.4,3.9,3.7,4.0,4.0,4.1,4.1,4.0,3.9,4.0,3.9,3.8,3.3,3.4,4.0,3.7,4.0,4.0,4.1,4.1,4.0,3.9,4.1,3.9,3.5,3.0,3.1,3.7,3.4,3.7,3.7,3.8,3.8,3.7,3.6,3.8,3.6
4,WEEKLY,1.9,1.8,1.9,2.0,1.9,2.0,1.9,2.0,2.0,2.0,2.0,2.0,2.0,1.9,1.9,1.9,2.0,1.9,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.6,1.5,1.5,1.6,1.5,1.6,1.6,1.7,1.7,1.6,1.6,1.7,1.7
5,UNKNOWN,1.2,1.2,1.2,1.2,1.2,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,0.6,0.6,0.6,0.6,0.6,0.7,0.6,0.7,0.7,0.7,0.7,0.7,0.8


In [15]:
cx_data1.to_clipboard()

### ps_tag_link

In [67]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317_link
    )
    
    
    select
        ps_tag_link ps_tag_link,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn1)
cx_data2.head()

,ps_tag_link,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,NPS,2.5,2.4,2.4,2.6,2.5,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.6,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.6,2.6,2.6,2.5,2.6,2.6,1.5,1.3,1.4,1.6,1.5,1.6,1.6,1.6,1.6,1.5,1.5,1.6,1.6
1,UNKNOWN,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.9,1.8,0.4,0.3,0.3,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4
2,PS,2.7,2.6,2.6,2.8,2.7,2.8,2.8,2.8,2.9,2.8,2.8,2.8,2.8,2.7,2.6,2.6,2.8,2.7,2.8,2.8,2.9,2.9,2.8,2.8,2.8,2.8,1.1,0.9,1.0,1.1,1.0,1.1,1.1,1.1,1.2,1.1,1.1,1.2,1.1


In [68]:
cx_data2

,ps_tag_link,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,NPS,2.5,2.4,2.4,2.6,2.5,2.6,2.5,2.6,2.6,2.5,2.5,2.6,2.6,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.6,2.6,2.6,2.5,2.6,2.6,1.5,1.3,1.4,1.6,1.5,1.6,1.6,1.6,1.6,1.5,1.5,1.6,1.6
1,UNKNOWN,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.9,1.8,0.4,0.3,0.3,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4
2,PS,2.7,2.6,2.6,2.8,2.7,2.8,2.8,2.8,2.9,2.8,2.8,2.8,2.8,2.7,2.6,2.6,2.8,2.7,2.8,2.8,2.9,2.9,2.8,2.8,2.8,2.8,1.1,0.9,1.0,1.1,1.0,1.1,1.1,1.1,1.2,1.1,1.1,1.2,1.1


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [69]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317_auto
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn1)
cx_data3.head()

,ps_tag_auto,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,PS,2.3,2.2,2.2,2.3,2.3,2.4,2.4,2.4,2.4,2.4,2.3,2.4,2.4,2.3,2.2,2.2,2.4,2.3,2.4,2.4,2.4,2.4,2.4,2.4,2.4,2.4,1.2,1.1,1.1,1.2,1.2,1.2,1.1,1.2,1.2,1.1,1.1,1.2,1.2
1,NPS,2.3,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.4,2.3,2.3,2.4,2.3,2.3,2.1,2.1,2.3,2.2,2.4,2.3,2.4,2.4,2.3,2.3,2.4,2.3,1.4,1.2,1.2,1.4,1.3,1.5,1.5,1.5,1.6,1.5,1.5,1.6,1.5
2,UNKNOWN,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.5,1.6,1.6,1.6,1.6,1.6,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.6,1.6,1.6,1.6,1.6,1.6,0.5,0.4,0.4,0.5,0.5,0.5,0.6,0.6,0.6,0.6,0.6,0.7,0.7


In [75]:
cx_data3

,ps_tag_auto,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,PS,2.3,2.2,2.2,2.3,2.3,2.4,2.4,2.4,2.4,2.4,2.3,2.4,2.4,2.3,2.2,2.2,2.4,2.3,2.4,2.4,2.4,2.4,2.4,2.4,2.4,2.4,1.2,1.1,1.1,1.2,1.2,1.2,1.1,1.2,1.2,1.1,1.1,1.2,1.2
1,NPS,2.3,2.1,2.1,2.3,2.2,2.3,2.3,2.3,2.4,2.3,2.3,2.4,2.3,2.3,2.1,2.1,2.3,2.2,2.4,2.3,2.4,2.4,2.3,2.3,2.4,2.3,1.4,1.2,1.2,1.4,1.3,1.5,1.5,1.5,1.6,1.5,1.5,1.6,1.5
2,UNKNOWN,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.5,1.6,1.6,1.6,1.6,1.6,1.4,1.3,1.3,1.4,1.4,1.5,1.5,1.6,1.6,1.6,1.6,1.6,1.6,0.5,0.4,0.4,0.5,0.5,0.5,0.6,0.6,0.6,0.6,0.6,0.7,0.7


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [33]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn1)
cx_data4.head()

,taxi_income_segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,MEDIUM_INCOME,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.3,1.4,1.4
1,LOW_INCOME,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4
2,UNKNOWN,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
3,HIGH_INCOME,1.9,1.8,1.8,1.9,1.9,2.0,1.9,2.0,2.0,1.9,1.9,2.0,2.0,1.9,1.8,1.8,2.0,1.9,2.0,1.9,2.0,2.0,2.0,1.9,2.0,2.0,1.5,1.4,1.4,1.5,1.5,1.6,1.5,1.6,1.6,1.5,1.5,1.6,1.6


In [37]:
cx_data4

,taxi_income_segment,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,MEDIUM_INCOME,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.3,1.4,1.4
1,LOW_INCOME,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4
2,UNKNOWN,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
3,HIGH_INCOME,1.9,1.8,1.8,1.9,1.9,2.0,1.9,2.0,2.0,1.9,1.9,2.0,2.0,1.9,1.8,1.8,2.0,1.9,2.0,1.9,2.0,2.0,2.0,1.9,2.0,2.0,1.5,1.4,1.4,1.5,1.5,1.6,1.5,1.6,1.6,1.5,1.5,1.6,1.6


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [39]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    )
    
    select
        taxi_service_affinity taxi_service_affinity,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn1)
cx_data5.head()

,taxi_service_affinity,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,ONLY_CAB,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.2,2.3,2.2,2.2,2.2,2.2,1.8,1.7,1.6,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.8,1.9,1.8
1,LINK_CAB,1.7,1.7,1.7,1.8,1.7,1.7,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.7,1.8,1.7,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.9,1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.4,1.3,1.3,1.4,1.4
2,ONLY_LINK,2.5,2.3,2.4,2.6,2.4,2.5,2.5,2.6,2.6,2.5,2.5,2.6,2.6,2.5,2.3,2.4,2.6,2.4,2.6,2.5,2.6,2.6,2.6,2.5,2.6,2.6,2.1,1.9,1.9,2.1,2.0,2.1,2.1,2.2,2.2,2.1,2.1,2.2,2.1
3,AUTO_CAB,2.1,2.0,2.0,2.1,2.1,2.2,2.1,2.2,2.2,2.2,2.1,2.2,2.2,2.1,2.0,2.0,2.1,2.1,2.2,2.1,2.2,2.2,2.2,2.2,2.2,2.2,1.8,1.7,1.6,1.8,1.7,1.9,1.8,1.9,1.9,1.9,1.8,1.9,1.9
4,NO_AFFINITY,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.7,2.7,2.6,2.6,2.7,2.7,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.7,2.7,2.7,2.7,2.7,2.7,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.3,2.3,2.3,2.3,2.3,2.3


In [40]:
cx_data5

,taxi_service_affinity,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,ONLY_CAB,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.2,2.3,2.2,2.2,2.2,2.2,1.8,1.7,1.6,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.8,1.9,1.8
1,LINK_CAB,1.7,1.7,1.7,1.8,1.7,1.7,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.7,1.8,1.7,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.9,1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.4,1.3,1.3,1.4,1.4
2,ONLY_LINK,2.5,2.3,2.4,2.6,2.4,2.5,2.5,2.6,2.6,2.5,2.5,2.6,2.6,2.5,2.3,2.4,2.6,2.4,2.6,2.5,2.6,2.6,2.6,2.5,2.6,2.6,2.1,1.9,1.9,2.1,2.0,2.1,2.1,2.2,2.2,2.1,2.1,2.2,2.1
3,AUTO_CAB,2.1,2.0,2.0,2.1,2.1,2.2,2.1,2.2,2.2,2.2,2.1,2.2,2.2,2.1,2.0,2.0,2.1,2.1,2.2,2.1,2.2,2.2,2.2,2.2,2.2,2.2,1.8,1.7,1.6,1.8,1.7,1.9,1.8,1.9,1.9,1.9,1.8,1.9,1.9
4,NO_AFFINITY,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.7,2.7,2.6,2.6,2.7,2.7,2.5,2.4,2.4,2.6,2.5,2.6,2.6,2.7,2.7,2.7,2.7,2.7,2.7,2.1,2.0,2.0,2.2,2.1,2.2,2.2,2.3,2.3,2.3,2.3,2.3,2.3
5,LINK_AUTO,2.2,2.0,2.1,2.2,2.1,2.2,2.2,2.2,2.3,2.2,2.2,2.3,2.3,2.2,2.1,2.1,2.2,2.1,2.2,2.2,2.3,2.3,2.2,2.2,2.3,2.3,1.8,1.6,1.7,1.8,1.7,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9
6,ONLY_AUTO,2.5,2.2,2.2,2.5,2.4,2.5,2.5,2.5,2.6,2.5,2.5,2.6,2.5,2.5,2.2,2.3,2.5,2.4,2.5,2.5,2.6,2.6,2.5,2.5,2.6,2.5,2.2,1.9,1.9,2.2,2.1,2.2,2.2,2.3,2.3,2.2,2.2,2.3,2.2
7,UNKNOWN,1.3,1.3,1.2,1.3,1.3,1.3,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,0.8,0.7,0.7,0.7,0.7,0.8,0.7,0.8,0.8,0.8,0.8,0.8,0.8


### RHA Check

In [43]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn1)
cx_data6.head()

,RHA_Check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,NRHA,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3
1,RHA,1.9,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,2.0,1.9,1.4,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
2,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8


In [44]:
cx_data6

,RHA_Check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,NRHA,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.6,1.6,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3
1,RHA,1.9,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,2.0,1.9,1.4,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
2,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8


### Zwigato_check

In [45]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn1)
cx_data7.head()

,Zwigato_check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,Zwigato,1.9,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.9,1.9,1.9,1.9,1.9,2.0,1.9,1.9,2.0,1.9,1.5,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
1,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
2,Non-Zwigato,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.8,1.8,1.2,1.1,1.2,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3


In [46]:
cx_data7

,Zwigato_check,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,Zwigato,1.9,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.9,1.9,1.9,1.9,1.9,2.0,1.9,1.9,2.0,1.9,1.5,1.3,1.3,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5,1.5,1.5
1,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
2,Non-Zwigato,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.7,1.8,1.8,1.2,1.1,1.2,1.2,1.2,1.2,1.2,1.3,1.3,1.2,1.2,1.3,1.3


### Office Usecases

In [47]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn1)
cx_data8.head()

,office_tag,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
1,no-office-apps,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.2,1.3,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.3,1.4,1.4
2,office-apps,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.7,1.7,1.9,1.8,1.9,1.9,2.0,2.0,1.9,1.9,2.0,1.9,1.5,1.3,1.3,1.5,1.4,1.5,1.5,1.6,1.6,1.5,1.5,1.6,1.5


In [48]:
cx_data8

,office_tag,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.1,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.2,2.1,2.1,2.2,2.1,1.7,1.5,1.5,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
1,no-office-apps,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.3,1.2,1.3,1.3,1.3,1.4,1.3,1.4,1.4,1.4,1.3,1.4,1.4
2,office-apps,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.7,1.7,1.9,1.8,1.9,1.9,2.0,2.0,1.9,1.9,2.0,1.9,1.5,1.3,1.3,1.5,1.4,1.5,1.5,1.6,1.6,1.5,1.5,1.6,1.5


### Age Group

In [51]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
     select week_start_date,
            customerid as customer_id,
            cast(ao_day as int)*1.0 as ao_day,
            cast(fe_day as int)*1.0 as fe_day,
            cast(rr_day as int)*1.0 as rr_day
     from hive.experiments_internal.customer_funnel_last90days_20250317
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        avg(case when week_start_date = '20241216' then ao_day end) w01_ao_ad_mean,
        avg(case when week_start_date = '20241223' then ao_day end ) w02_ao_ad_mean,
        avg(case when week_start_date = '20241230' then ao_day end ) w03_ao_ad_mean,
        avg(case when week_start_date = '20250106' then ao_day end ) w04_ao_ad_mean,
        avg(case when week_start_date = '20250113' then ao_day end ) w05_ao_ad_mean,
        avg(case when week_start_date = '20250120' then ao_day end ) w06_ao_ad_mean,
        avg(case when week_start_date = '20250127' then ao_day end ) w07_ao_ad_mean,
        avg(case when week_start_date = '20250203' then ao_day end ) w08_ao_ad_mean,
        avg(case when week_start_date = '20250210' then ao_day end ) w09_ao_ad_mean,
        avg(case when week_start_date = '20250217' then ao_day end ) w10_ao_ad_mean,
        avg(case when week_start_date = '20250224' then ao_day end ) w11_ao_ad_mean,
        avg(case when week_start_date = '20250303' then ao_day end ) w12_ao_ad_mean,
        avg(case when week_start_date = '20250310' then ao_day end ) w13_ao_ad_mean,
        
        avg(case when week_start_date = '20241216' then fe_day end ) w01_fe_ad_mean,
        avg(case when week_start_date = '20241223' then fe_day end ) w02_fe_ad_mean,
        avg(case when week_start_date = '20241230' then fe_day end ) w03_fe_ad_mean,
        avg(case when week_start_date = '20250106' then fe_day end ) w04_fe_ad_mean,
        avg(case when week_start_date = '20250113' then fe_day end ) w05_fe_ad_mean,
        avg(case when week_start_date = '20250120' then fe_day end ) w06_fe_ad_mean,
        avg(case when week_start_date = '20250127' then fe_day end ) w07_fe_ad_mean,
        avg(case when week_start_date = '20250203' then fe_day end ) w08_fe_ad_mean,
        avg(case when week_start_date = '20250210' then fe_day end ) w09_fe_ad_mean,
        avg(case when week_start_date = '20250217' then fe_day end ) w10_fe_ad_mean,
        avg(case when week_start_date = '20250224' then fe_day end ) w11_fe_ad_mean,
        avg(case when week_start_date = '20250303' then fe_day end ) w12_fe_ad_mean,
        avg(case when week_start_date = '20250310' then fe_day end ) w13_fe_ad_mean,

        avg(case when week_start_date = '20241216' then rr_day end ) w01_rr_ad_mean,
        avg(case when week_start_date = '20241223' then rr_day end ) w02_rr_ad_mean,
        avg(case when week_start_date = '20241230' then rr_day end ) w03_rr_ad_mean,
        avg(case when week_start_date = '20250106' then rr_day end ) w04_rr_ad_mean,
        avg(case when week_start_date = '20250113' then rr_day end ) w05_rr_ad_mean,
        avg(case when week_start_date = '20250120' then rr_day end ) w06_rr_ad_mean,
        avg(case when week_start_date = '20250127' then rr_day end ) w07_rr_ad_mean,
        avg(case when week_start_date = '20250203' then rr_day end ) w08_rr_ad_mean,
        avg(case when week_start_date = '20250210' then rr_day end ) w09_rr_ad_mean,
        avg(case when week_start_date = '20250217' then rr_day end ) w10_rr_ad_mean,
        avg(case when week_start_date = '20250224' then rr_day end ) w11_rr_ad_mean,
        avg(case when week_start_date = '20250303' then rr_day end ) w12_rr_ad_mean,
        avg(case when week_start_date = '20250310' then rr_day end ) w13_rr_ad_mean

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn1)
cx_data9.head(10)

,age_group,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,21-25,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.9,1.8,1.8,1.9,1.9,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.9,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4
1,26-30,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.4,1.4,1.5,1.5,1.4,1.4,1.5,1.5
2,31-35,1.8,1.7,1.7,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
3,36-45,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
4,Above-45,1.8,1.7,1.7,1.9,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
5,Below-21,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.4,1.3,1.3,1.3,1.4,1.4
6,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.0,2.0,2.0,1.9,2.0,2.0,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.0,2.0,2.0,2.0,2.0,1.7,1.5,1.5,1.7,1.6,1.8,1.7,1.7,1.6,1.6,1.6,1.6,1.6


In [52]:
cx_data9

,age_group,w01_ao_ad_mean,w02_ao_ad_mean,w03_ao_ad_mean,w04_ao_ad_mean,w05_ao_ad_mean,w06_ao_ad_mean,w07_ao_ad_mean,w08_ao_ad_mean,w09_ao_ad_mean,w10_ao_ad_mean,w11_ao_ad_mean,w12_ao_ad_mean,w13_ao_ad_mean,w01_fe_ad_mean,w02_fe_ad_mean,w03_fe_ad_mean,w04_fe_ad_mean,w05_fe_ad_mean,w06_fe_ad_mean,w07_fe_ad_mean,w08_fe_ad_mean,w09_fe_ad_mean,w10_fe_ad_mean,w11_fe_ad_mean,w12_fe_ad_mean,w13_fe_ad_mean,w01_rr_ad_mean,w02_rr_ad_mean,w03_rr_ad_mean,w04_rr_ad_mean,w05_rr_ad_mean,w06_rr_ad_mean,w07_rr_ad_mean,w08_rr_ad_mean,w09_rr_ad_mean,w10_rr_ad_mean,w11_rr_ad_mean,w12_rr_ad_mean,w13_rr_ad_mean
0,21-25,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.9,1.8,1.8,1.9,1.9,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.9,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.3,1.4,1.4
1,26-30,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.4,1.4,1.5,1.5,1.4,1.4,1.5,1.5
2,31-35,1.8,1.7,1.7,1.8,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
3,36-45,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
4,Above-45,1.8,1.7,1.7,1.9,1.8,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.8,1.7,1.7,1.9,1.8,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.4,1.3,1.3,1.4,1.4,1.5,1.4,1.5,1.5,1.5,1.5,1.5,1.5
5,Below-21,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.7,1.7,1.8,1.8,1.8,1.8,1.9,1.9,1.8,1.8,1.9,1.9,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.4,1.3,1.3,1.3,1.4,1.4
6,UNKNOWN,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.0,2.0,2.0,1.9,2.0,2.0,2.0,1.9,1.9,2.1,2.0,2.1,2.1,2.1,2.0,2.0,2.0,2.0,2.0,1.7,1.5,1.5,1.7,1.6,1.8,1.7,1.7,1.6,1.6,1.6,1.6,1.6


### Office Use-Cases

In [61]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73


In [62]:
cx_data11

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73
